# Date Ranges, Frequencies, and Shifting

The previous notebook's time series were **irregular**: no fixed spacing between timestamps. That's fine for many applications, but it's often desirable to work relative to a fixed frequency — daily, monthly, every 15 minutes — even if that means introducing missing values where data doesn't actually exist. pandas has a full suite of standard time series frequencies and tools for working with them.

As a preview (the deep dive is in the *Resampling and Frequency Conversion* notebook later in this chapter), here's converting our sample time series to a fixed daily frequency with `resample`:

In [1]:
from datetime import datetime 
import numpy as np 
import pandas as pd 

dates = [datetime(2011, 1, 2), datetime(2011, 1, 5), datetime(2011, 1, 7), datetime(2011, 1, 8), datetime(2011, 1, 10), datetime(2011, 1, 12)]
    
ts = pd.Series(np.random.standard_normal(6), index=dates)

ts

2011-01-02   -0.207686
2011-01-05   -0.675223
2011-01-07    1.305470
2011-01-08    2.844317
2011-01-10    0.757319
2011-01-12   -0.315775
dtype: float64

In [2]:
resampler = ts.resample("D")

resampler

The string `"D"` is interpreted as daily frequency. Notice `resampler` didn't actually compute anything — this should look familiar: a `Resampler` is **lazy**, exactly like a `GroupBy` object (Chapter 10). Nothing happens until you call an aggregation like `.mean()` on it. (Again, more on this in the *Resampling* notebook — the point here is just that frequencies show up everywhere in this chapter, including tools we haven't formally covered yet.)

## Generating Date Ranges

`pandas.date_range` is responsible for generating a `DatetimeIndex` of an indicated length according to a particular frequency:

By default, `date_range` produces timestamps by adding multiples of the offset between start and end, inclusive of both endpoints:

In [3]:
index = pd.date_range("2012-04-01", "2012-06-01")

index

DatetimeIndex(['2012-04-01', '2012-04-02', '2012-04-03', '2012-04-04',
               '2012-04-05', '2012-04-06', '2012-04-07', '2012-04-08',
               '2012-04-09', '2012-04-10', '2012-04-11', '2012-04-12',
               '2012-04-13', '2012-04-14', '2012-04-15', '2012-04-16',
               '2012-04-17', '2012-04-18', '2012-04-19', '2012-04-20',
               '2012-04-21', '2012-04-22', '2012-04-23', '2012-04-24',
               '2012-04-25', '2012-04-26', '2012-04-27', '2012-04-28',
               '2012-04-29', '2012-04-30', '2012-05-01', '2012-05-02',
               '2012-05-03', '2012-05-04', '2012-05-05', '2012-05-06',
               '2012-05-07', '2012-05-08', '2012-05-09', '2012-05-10',
               '2012-05-11', '2012-05-12', '2012-05-13', '2012-05-14',
               '2012-05-15', '2012-05-16', '2012-05-17', '2012-05-18',
               '2012-05-19', '2012-05-20', '2012-05-21', '2012-05-22',
               '2012-05-23', '2012-05-24', '2012-05-25', '2012-05-26',
      

By default, `pandas.date_range` generates daily timestamps. If you pass only a start or end date, you must pass a number of `periods` to generate. Notice the direction each one counts: `start` + `periods` counts *forward* from the start date, while `end` + `periods` counts *backward* from the end date:

In [4]:
pd.date_range(start="2012-04-01", periods=20)

DatetimeIndex(['2012-04-01', '2012-04-02', '2012-04-03', '2012-04-04',
               '2012-04-05', '2012-04-06', '2012-04-07', '2012-04-08',
               '2012-04-09', '2012-04-10', '2012-04-11', '2012-04-12',
               '2012-04-13', '2012-04-14', '2012-04-15', '2012-04-16',
               '2012-04-17', '2012-04-18', '2012-04-19', '2012-04-20'],
              dtype='datetime64[us]', freq='D')

In [5]:
pd.date_range(end="2012-06-01", periods=20)

DatetimeIndex(['2012-05-13', '2012-05-14', '2012-05-15', '2012-05-16',
               '2012-05-17', '2012-05-18', '2012-05-19', '2012-05-20',
               '2012-05-21', '2012-05-22', '2012-05-23', '2012-05-24',
               '2012-05-25', '2012-05-26', '2012-05-27', '2012-05-28',
               '2012-05-29', '2012-05-30', '2012-05-31', '2012-06-01'],
              dtype='datetime64[us]', freq='D')

The start and end dates define strict boundaries for the generated date index — only dates falling on or inside the interval are included. For example, if you wanted a date index containing the last business day of each month, you'd pass the `"BME"` frequency (business month end; see the full alias table below):

In [6]:
pd.date_range("2000-01-01", "2000-12-01", freq="BME")

DatetimeIndex(['2000-01-31', '2000-02-29', '2000-03-31', '2000-04-28',
               '2000-05-31', '2000-06-30', '2000-07-31', '2000-08-31',
               '2000-09-29', '2000-10-31', '2000-11-30'],
              dtype='datetime64[us]', freq='BME')

## Frequency Aliases

> **Nuance — this is probably the single biggest source of confusion in this chapter.** Older pandas (and most existing tutorials, Stack Overflow answers, and even older editions of the book this material is based on) use single-letter aliases like `M`, `BM`, `A`, `Q`, `T`, `L`, `U`, `H`. **As of pandas 2.2+, these don't just warn — they raise a hard `ValueError`.** If you paste an old snippet and get `ValueError: 'M' is no longer supported for offsets. Please use 'ME' instead.`, this is why. Always use the **current alias** column below; the old-alias column is there so you recognize what you're looking at when you see it in older material.

| Current alias | Old alias (now an error) | Offset type | Description |
|---|---|---|---|
| `D` | — | `Day` | Calendar daily |
| `B` | — | `BusinessDay` | Business daily |
| `h` | `H` | `Hour` | Hourly |
| `min` | `T` | `Minute` | Minutely |
| `s` | `S` | `Second` | Secondly |
| `ms` | `L` | `Milli` | Millisecond |
| `us` | `U` | `Micro` | Microsecond |
| `ME` | `M` | `MonthEnd` | Last calendar day of month |
| `BME` | `BM` | `BusinessMonthEnd` | Last business day of month |
| `MS` | — | `MonthBegin` | First calendar day of month |
| `BMS` | — | `BusinessMonthBegin` | First business day of month |
| `W-MON`, `W-TUE`, ... | — | `Week` | Weekly on given day of week (MON, TUE, WED, THU, FRI, SAT, or SUN) |
| `WOM-3FRI` | — | `WeekOfMonth` | Weekly dates in the 1st–4th week of the month (e.g., `WOM-3FRI` for the third Friday of each month) |
| `QE-JAN`, `QE-FEB`, ... | `Q-JAN`, ... | `QuarterEnd` | Quarterly, anchored on last calendar day of each month, for a fiscal year ending in the indicated month |
| `BQE-JAN`, `BQE-FEB`, ... | `BQ-JAN`, ... | `BusinessQuarterEnd` | Same, anchored on last business day |
| `QS-JAN`, `QS-FEB`, ... | — | `QuarterBegin` | Quarterly, anchored on first calendar day of each month |
| `BQS-JAN`, `BQS-FEB`, ... | — | `BusinessQuarterBegin` | Same, anchored on first business day |
| `YE-JAN`, `YE-FEB`, ... | `A-JAN`, ... | `YearEnd` | Annual, anchored on last calendar day of given month |
| `BYE-JAN`, `BYE-FEB`, ... | `BA-JAN`, ... | `BusinessYearEnd` | Same, anchored on last business day |
| `YS-JAN`, `YS-FEB`, ... | `AS-JAN`, ... | `YearBegin` | Annual, anchored on first day of given month |
| `BYS-JAN`, `BYS-FEB`, ... | `BAS-JAN`, ... | `BusinessYearBegin` | Same, anchored on first business day |

Every "end" alias has a mirror-image "begin" alias (`ME`/`MS`, `QE`/`QS`, `YE`/`YS`, and their business-day `B`-prefixed variants) — that consistent E/S suffix pattern is the easiest way to remember the modern names.

`pandas.date_range` by default preserves the time (if any) of the start or end time-stamp:

In [7]:
pd.date_range("2012-05-02 12:56:31", periods=5)

DatetimeIndex(['2012-05-02 12:56:31', '2012-05-03 12:56:31',
               '2012-05-04 12:56:31', '2012-05-05 12:56:31',
               '2012-05-06 12:56:31'],
              dtype='datetime64[us]', freq='D')

Sometimes you will have start or end dates with time information but want to generate a set of timestamps *normalized* to midnight as a convention. To do this, there is a `normalize` option:

In [8]:
pd.date_range("2012-05-02 12:56:31", periods=5, normalize=True)

DatetimeIndex(['2012-05-02', '2012-05-03', '2012-05-04', '2012-05-05',
               '2012-05-06'],
              dtype='datetime64[us]', freq='D')

## Frequencies and Date Offsets

Frequencies in pandas are composed of a *base frequency* and a multiplier. Base frequencies are typically referred to by a string alias, like `"ME"` for month-end or `"h"` for hourly (see the table above). For each base frequency there's a corresponding **date offset** object. For example, hourly frequency is represented by the `Hour` class:

In [9]:
from pandas.tseries.offsets import Hour, Minute

hour = Hour()

hour

<Hour>

You can define a multiple of an offset by passing an integer:

In [10]:
four_hours = Hour(4)

four_hours

<4 * Hours>

In most applications, you'd never need to explicitly create one of these objects; instead you'd use a string alias like `"h"` or `"4h"`. Putting an integer before the base frequency creates a multiple:

In [11]:
pd.date_range("2000-01-01", "2000-01-03 23:59", freq="4h" )

DatetimeIndex(['2000-01-01 00:00:00', '2000-01-01 04:00:00',
               '2000-01-01 08:00:00', '2000-01-01 12:00:00',
               '2000-01-01 16:00:00', '2000-01-01 20:00:00',
               '2000-01-02 00:00:00', '2000-01-02 04:00:00',
               '2000-01-02 08:00:00', '2000-01-02 12:00:00',
               '2000-01-02 16:00:00', '2000-01-02 20:00:00',
               '2000-01-03 00:00:00', '2000-01-03 04:00:00',
               '2000-01-03 08:00:00', '2000-01-03 12:00:00',
               '2000-01-03 16:00:00', '2000-01-03 20:00:00'],
              dtype='datetime64[us]', freq='4h')

Many offsets can be combined by addition — pandas reduces them to a common unit automatically (here, hours get converted to minutes):

In [12]:
Hour(2) + Minute(30)

<150 * Minutes>

Similarly, you can pass frequency strings, like `"1h30min"`, that get parsed to the same expression:

In [13]:
pd.date_range("2000-01-01", periods=10, freq="1h30min")

DatetimeIndex(['2000-01-01 00:00:00', '2000-01-01 01:30:00',
               '2000-01-01 03:00:00', '2000-01-01 04:30:00',
               '2000-01-01 06:00:00', '2000-01-01 07:30:00',
               '2000-01-01 09:00:00', '2000-01-01 10:30:00',
               '2000-01-01 12:00:00', '2000-01-01 13:30:00'],
              dtype='datetime64[us]', freq='90min')

### Week of Month

One useful frequency class is "week of month," aliased `WOM`. This lets you get dates like the third Friday of each month. The pattern is `WOM-<week#><weekday>`, where `<week#>` is 1–4 and `<weekday>` is the same three-letter code used by `W-MON`, etc. — so `WOM-3FRI` means "the 3rd Friday of the month":

In [14]:
monthly_date = pd.date_range("2012-01-01", "2012-09-01", freq="WOM-3FRI")

list(monthly_date)

[Timestamp('2012-01-20 00:00:00'),
 Timestamp('2012-02-17 00:00:00'),
 Timestamp('2012-03-16 00:00:00'),
 Timestamp('2012-04-20 00:00:00'),
 Timestamp('2012-05-18 00:00:00'),
 Timestamp('2012-06-15 00:00:00'),
 Timestamp('2012-07-20 00:00:00'),
 Timestamp('2012-08-17 00:00:00')]

## Shifting (Leading and Lagging) Data

**Shifting** refers to moving data backward and forward through time, without changing the calendar dates any other data lives on. The terminology comes from finance/statistics:
- **Lag** a series: shift it *forward* in time (`shift(n)` with `n > 0`) — today's row now holds what used to be *n* periods ago. Useful for "what was the value last month?" comparisons.
- **Lead** a series: shift it *backward* in time (`shift(n)` with `n < 0`) — today's row now holds what's coming *n* periods from now.

Both Series and DataFrame have a `shift` method. There are two distinct ways to use it, and mixing them up is the most common source of confusion in this section:

1. **Naive shift** (default): the *data* moves to different positions, but the *index stays exactly as it was*. This creates `NaN`s wherever data got shifted off the edge, because there's no longer a value for that original timestamp.
2. **Frequency-aware shift** (pass `freq=`): the *index labels themselves* move forward/backward by that amount, while each data value stays attached to whatever it started attached to. No `NaN`s are introduced — you get a *relabeled* series, not a series with holes.

Let's see the naive version first:

In [15]:
ts = pd.Series(np.random.standard_normal(4), index= pd.date_range("2000-01-01", periods=4, freq="ME"))

ts

2000-01-31    0.450016
2000-02-29    1.102074
2000-03-31   -1.008154
2000-04-30   -0.096795
Freq: ME, dtype: float64

Compare `ts` above with `ts.shift(2)` below — same four index labels, but every value has moved down two rows, pushing the last two values off the end entirely and leaving `NaN` at the top:

In [16]:
ts.shift(2)

2000-01-31         NaN
2000-02-29         NaN
2000-03-31    0.450016
2000-04-30    1.102074
Freq: ME, dtype: float64

Notice the index (`2000-01-31` through `2000-04-30`) is unchanged from `ts` above — only the data moved, leaving `NaN` at the start.

A common use of a naive `shift` is computing consecutive percent changes in a time series (or several, as DataFrame columns):

In [17]:
ts / ts.shift(1) - 1

2000-01-31         NaN
2000-02-29    1.448966
2000-03-31   -1.914779
2000-04-30   -0.903987
Freq: ME, dtype: float64

This pattern is common enough that pandas gives you a shortcut: `ts.pct_change()` computes exactly the same thing directly.

Now the frequency-aware version. Passing `freq="ME"` tells `shift` to advance every *timestamp* by 2 month-ends instead of shuffling the *data* — compare the index below (now `2000-03-31` through `2000-06-30`) to the unchanged index we saw with the naive shift above:

In [18]:
ts.shift(2, freq="ME")

2000-03-31    0.450016
2000-04-30    1.102074
2000-05-31   -1.008154
2000-06-30   -0.096795
Freq: ME, dtype: float64

You're not limited to the series' own frequency — any offset alias works, which is what makes `freq=` shifts flexible. Here we shift by 3 calendar days instead of 3 months. Notice the result has *no* `Freq: ME` label anymore, since shifting `ME`-anchored timestamps by a `D`-based offset moves them off their monthly anchors:

In [19]:
ts.shift(3, freq="D")

2000-02-03    0.450016
2000-03-03    1.102074
2000-04-03   -1.008154
2000-05-03   -0.096795
dtype: float64

In [20]:
ts.shift(1, freq="90min")

2000-01-31 01:30:00    0.450016
2000-02-29 01:30:00    1.102074
2000-03-31 01:30:00   -1.008154
2000-04-30 01:30:00   -0.096795
dtype: float64

## Using Date Offsets with `groupby`

A creative use of date offsets is combining them with `groupby` to bucket irregular timestamps into a coarser period — a manual precursor to resampling. The key piece is a date offset's `rollforward` method: it rounds a date *up* to the next occurrence of that offset (or leaves it unchanged if it's already on one). `MonthEnd().rollforward(ts)` maps every timestamp to *the month-end that follows it* — so all dates in January map to `2000-01-31`, all dates in February map to `2000-02-29`, and so on, which is exactly what you want as a `groupby` key:

In [21]:
ts = pd.Series(np.random.standard_normal(20), index=pd.date_range("2000-01-15", periods=20, freq="4D"))

ts

2000-01-15    0.293680
2000-01-19    1.435467
2000-01-23    0.986837
2000-01-27   -1.136608
2000-01-31   -0.312332
2000-02-04    1.656983
2000-02-08    1.605034
2000-02-12   -2.506760
2000-02-16    0.475784
2000-02-20    1.400338
2000-02-24   -0.642124
2000-02-28    1.703601
2000-03-03    1.218486
2000-03-07   -0.848506
2000-03-11   -0.092329
2000-03-15   -0.442928
2000-03-19    0.126377
2000-03-23   -1.090807
2000-03-27   -0.722695
2000-03-31    0.449570
Freq: 4D, dtype: float64

In [22]:
from pandas.tseries.offsets import Day, MonthEnd

ts.groupby(MonthEnd().rollforward).mean()

2000-01-31    0.253409
2000-02-29    0.527551
2000-03-31   -0.175354
dtype: float64

A faster (and far more common) way to accomplish the same grouping is resampling — the manual `groupby(MonthEnd().rollforward)` above and this `resample("ME")` produce identical results, but `resample` is purpose-built for it. This is exactly the tool previewed at the very start of this notebook; the *Resampling and Frequency Conversion* notebook covers it properly:

In [23]:
ts.resample("ME").mean()

2000-01-31    0.253409
2000-02-29    0.527551
2000-03-31   -0.175354
Freq: ME, dtype: float64

---

## Summary / Cheat Sheet

**Generating date ranges:**

| What you want | How |
|---|---|
| Every day between two dates | `pd.date_range(start, end)` |
| N days starting from a date | `pd.date_range(start=start, periods=n)` |
| N days ending on a date | `pd.date_range(end=end, periods=n)` |
| Last business day of each month | `pd.date_range(start, end, freq="BME")` |
| Timestamps snapped to midnight | `pd.date_range(..., normalize=True)` |
| 3rd Friday of each month | `pd.date_range(start, end, freq="WOM-3FRI")` |

**Frequency aliases:** use the *current* alias, not the one-letter alias from older tutorials — `M`/`BM`/`A`/`Q`/`T`/`L`/`U`/`H` are now hard errors. Reach for `ME`/`BME`/`YE`/`QE`/`min`/`ms`/`us`/`h` instead (full table above).

**`shift()` — the two modes:**

| | Naive shift (default) | Frequency-aware shift (`freq=`) |
|---|---|---|
| What moves | The data | The index labels |
| Index after shifting | Unchanged | Shifted by `n × freq` |
| Introduces `NaN`? | Yes, at whichever end data fell off | No |
| Typical use | `ts / ts.shift(1) - 1` (or just `ts.pct_change()`) | "What date was this reading actually taken 3 days earlier?" — relabeling, not comparing |

**Nuances worth remembering:**
- `shift(n)` with `n > 0` **lags** a series (today holds what happened *n* periods ago); `n < 0` **leads** it (today holds what's coming).
- A date offset's `.rollforward()` / `.rollback()` round a timestamp up/down to the next/previous occurrence of that offset — the trick behind bucketing irregular timestamps with `groupby`, and conceptually what `resample` does for you automatically.
- `Resampler` objects (from `.resample(...)`) are lazy, just like `GroupBy` objects — nothing computes until you call an aggregation.

**Up next:** *Time Zone Handling* — localizing naive timestamps to a time zone, converting between zones, and how arithmetic behaves across zone-aware `Timestamp` objects.